# PARS FarmAnchor Absolute-Anchor Exploration
### State-consistent robust farm/global anchor; original PARS remains the reference

This notebook is the compact, reproducible entry point for the **PARS FarmAnchor exploration candidate**.

This notebook is an absolute-anchor exploration variant of the released **Physics-Anchored Relative-State (PARS)** model. It does not replace `YawMisalignment_Independent_Model_Release.ipynb`. The relative-state pipeline remains frozen; only the shared absolute centre `C` is changed.

For labelled turbine `i` and day `d`, the state-consistent anchor observation is

$$
a_i(d)=y_i(d)+c_{i,0.75}^{\mathrm{soft}}(d)+\theta_i^*.
$$

Daily observations are reduced to a quality-weighted turbine-level arithmetic mean. A robust Huber centre across those turbine means supplies `C`; a same-farm centre is preferred when labelled support exists, otherwise the global centre is used.

$$
A_i=\frac{\sum_d q_i(d)a_i(d)}{\sum_d q_i(d)}.
$$

The final exploration prediction is

$$
\hat y_i(d)=C_{\mathrm{corrected}}-\theta_i^*-c_{i,0.75}^{\mathrm{soft}}(d),\qquad \beta=1$$.

Here, \($\theta_i^{*}$\) is the unchanged global median of rolling power-vs-vane argmax estimates, while \($c_{i,0.75}^{\mathrm{soft}}(d)$\) is the unchanged soft, partial-amplitude relative-heading correction. Power is used only in the SCADA-derived `theta_star` absolute calibration; it is not a day-level dynamic predictor or an input to the relative-state correction.

### Current validation result

Strict development-turbine LOTO reference:

| Anchor | Macro MAE | Macro RMSE |
|---|---:|---:|
| Release PARS anchor | 0.358 | 0.524 |
| FarmAnchor corrected centre | **0.320** | **0.510** |

The only model change relative to `YawMisalignment_Independent_Model_Release.ipynb` is the absolute anchor: the exploration candidate estimates daily state-consistent labelled support `a_i(d)=y_i(d)+c_i(d)+theta_star_i`, forms a quality-weighted turbine-level arithmetic mean, and then applies a robust Huber centre across turbines. Relative-heading boundaries, fixed pair/sector baselines, soft weighting, `lambda=0.75`, `beta=1`, and the release notebook remain unchanged.

Full-fit corrected anchor: `C_corrected = -6.155553°`. The PPP development set contains one farm, so this is not yet a validated cross-farm transfer result. No external leaderboard validation has been performed.

The final section contains an optional submission export; it is disabled only by choosing a different target or anchor setting.

## 1. Reproducibility and data split

The validation split is strict leave-one-turbine-out (LOTO):

- labelled development turbines: `PPP_WTG12`, `PPP_WTG13`, `PPP_WTG14`;
- unlabeled targets inspected after model fitting: `PPP_WTG17`, `SSS_WTG06`;
- all published turbines may contribute **unlabeled** same-site context to the state detector.

The target/holdout labels are never used to construct their SCADA states.

In [1]:
from pathlib import Path
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# Find repository root without a machine-specific absolute path.
HERE = Path.cwd().resolve()
ROOT = HERE

if not (ROOT / "src").exists():
    matches = [
        parent
        for parent in [HERE, *HERE.parents]
        if (parent / "src").exists()
    ]
    if not matches:
        raise FileNotFoundError(
            "Could not locate repository root containing src/."
        )
    ROOT = matches[0]

zip_candidates = [
    ROOT / "turbines_data.zip",
    ROOT.parent / "turbines_data.zip",
]
ZIP_PATH = next(
    (path for path in zip_candidates if path.exists()),
    None,
)

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Place turbines_data.zip in the repository root or its parent directory."
    )

sys.path.insert(0, str(ROOT / "src"))

from baseline_loto_ridge import read_turbine, list_turbines, wrap_180
from fleet_context import (
    FleetConfig,
    load_layout,
    bin_turbine,
    circular_median_deg,
)
from yaw_relative_state import RelativeStateConfig
from fleet_context import circular_mean_deg
from yaw_farm_anchor import (
    build_anchor_observations,
    fit_farm_anchor,
    legacy_global_anchor,
    turbine_farm_key,
)
from yaw_theta_stability import (
    bootstrap_theta_locations,
    corrected_mean_anchor_center,
    run_theta_location_loto,
    theta_location_estimates,
    run_theta_anchor_loto,
    stabilize_theta_star_map,
    estimate_theta_star_series,
)
from yaw_model import (
    ModelConfig,
    build_turbine_bundle,
    apply_site_common_mode_veto,
    fit_global_beta,
    predict_from_bundle,
    score_prediction,
)

TRAIN = ["PPP_WTG12", "PPP_WTG13", "PPP_WTG14"]
TARGETS = ["PPP_WTG17", "SSS_WTG06"]

FLEET = FleetConfig()
RELATIVE = RelativeStateConfig()
MODEL = ModelConfig()

print("Repository:", ROOT)
print("SCADA archive:", ZIP_PATH)

Repository: E:\EnergyHacks\github_release
SCADA archive: E:\EnergyHacks\turbines_data.zip


## 2. Reference PARS anchor: long-term aerodynamic reference

This section reproduces the release PARS absolute prior for comparison. The FarmAnchor variant changes only the shared `C` estimate later in the notebook; the turbine-specific `theta_star` estimator is unchanged.

For each turbine, the frozen B0 estimator computes a rolling apparent power-optimal vane angle
\($\hat\theta_{\text{argmax}}(t)$\). The long-term turbine-specific reference is

$$
\theta_i^{*} = \mathrm{median}_t \hat\theta_{\text{argmax},i}(t).
$$

In the release model, the labelled training turbines calibrate one shared absolute offset:

$$
C = \mathrm{mean}_{i\in\text{train}} \left(\bar y_i + \theta_i^{*}\right).
$$

The deployed default is therefore

$$
B0_i=C-\theta_i^{*}.
$$

Dynamic state logic is not allowed to replace this absolute anchor; it can only make sparse corrections around it.

In [2]:
t0 = time.perf_counter()

progress_t0 = time.perf_counter()
turbines = list_turbines(str(ZIP_PATH))
print(f'Found {len(turbines)} turbines. Loading raw data...', flush=True)
raw = {}
for i, turbine in enumerate(turbines, start=1):
    raw[turbine] = read_turbine(str(ZIP_PATH), turbine)
    print(f'  raw {i}/{len(turbines)}: {turbine}', flush=True)

layout = load_layout(ROOT, turbines)
print(f'Layout loaded for {len(layout)} turbines.', flush=True)

print('Binning turbine data...', flush=True)
binned = {}
for i, turbine in enumerate(turbines, start=1):
    binned[turbine] = bin_turbine(raw[turbine], FLEET)
    print(f'  binned {i}/{len(turbines)}: {turbine}', flush=True)

labels = {
    turbine: (
        raw[turbine]
        .assign(date=pd.to_datetime(raw[turbine]["date"]))
        .groupby("date")["yaw_misalignment_deg"]
        .median()
        .sort_index()
    )
    for turbine in TRAIN
}

print('Estimating rolling theta_star for labelled and target turbines...', flush=True)
theta_series = {}
theta_star = {}
for i, turbine in enumerate(TRAIN + TARGETS, start=1):
    theta_series[turbine] = estimate_theta_star_series(raw[turbine])
    theta_star[turbine] = float(theta_series[turbine].dropna().median())
    print(
        f'  theta {i}/{len(TRAIN + TARGETS)}: {turbine}; '
        f'valid={theta_series[turbine].notna().sum()}/{len(theta_series[turbine])}; '
        f'median={theta_star[turbine]:.3f}',
        flush=True,
    )
print(f'theta_star ready. Elapsed: {time.perf_counter() - progress_t0:.1f}s', flush=True)

LOAD_SECONDS = time.perf_counter() - t0

theta_table = (
    pd.Series(theta_star, name="theta_star_deg")
    .rename_axis("turbine")
    .to_frame()
)

display(theta_table.round(3))
print(f"Load + B0 estimation: {LOAD_SECONDS:.1f}s")

Found 16 turbines. Loading raw data...
  raw 1/16: PPP_WTG07
  raw 2/16: PPP_WTG08
  raw 3/16: PPP_WTG11
  raw 4/16: PPP_WTG12
  raw 5/16: PPP_WTG13
  raw 6/16: PPP_WTG14
  raw 7/16: PPP_WTG15
  raw 8/16: PPP_WTG16
  raw 9/16: PPP_WTG17
  raw 10/16: PPP_WTG18
  raw 11/16: PPP_WTG33
  raw 12/16: SSS_WTG04
  raw 13/16: SSS_WTG05
  raw 14/16: SSS_WTG06
  raw 15/16: SSS_WTG07
  raw 16/16: SSS_WTG16
Layout loaded for 16 turbines.
Binning turbine data...
  binned 1/16: PPP_WTG07
  binned 2/16: PPP_WTG08
  binned 3/16: PPP_WTG11
  binned 4/16: PPP_WTG12
  binned 5/16: PPP_WTG13
  binned 6/16: PPP_WTG14
  binned 7/16: PPP_WTG15
  binned 8/16: PPP_WTG16
  binned 9/16: PPP_WTG17
  binned 10/16: PPP_WTG18
  binned 11/16: PPP_WTG33
  binned 12/16: SSS_WTG04
  binned 13/16: SSS_WTG05
  binned 14/16: SSS_WTG06
  binned 15/16: SSS_WTG07
  binned 16/16: SSS_WTG16
Estimating rolling theta_star for labelled and target turbines...
  theta 1/5: PPP_WTG12; valid=698/731; median=-3.444
  theta 2/5: PPP_WTG1

,theta_star_deg
turbine,
PPP_WTG12,-3.444
PPP_WTG13,0.352
PPP_WTG14,-4.354
PPP_WTG17,-2.483
SSS_WTG06,-1.513


Load + B0 estimation: 173.4s


## 3. Label-free relative-heading state detector

The dynamic channel uses only SCADA and same-site turbine context.

For target \(i\) and neighbour \(j\):

1. form circular heading differences;
2. remove pair/sector baselines;
3. estimate pair reliability from coverage, residual MAD and distance;
4. robustly aggregate usable pairs into a daily target-relative heading residual;
5. construct a neighbour-only background residual to identify common-mode motion.

Two complementary change detectors are used on the cleaned relative signal:

- a persistent rolling before/after detector;
- a conservative L2 state segmentation used only for long two-sided regimes.

A candidate is rejected when it is better explained by:

- sensor/encoder-like jumps;
- the configured sensor-shadow window;
- local background/common-mode motion;
- insufficient pair agreement;
- same-site synchronous events;
- weak event confidence.

Only surviving high-confidence events are allowed to move the prediction away from the constant prior.

In [3]:
t0 = time.perf_counter()

# Build every published turbine so the site-common-mode veto uses the full
# unlabeled site context rather than only the labelled turbines.
bundle_t0 = time.perf_counter()
print(f'Building relative-heading bundles for {len(turbines)} turbines...', flush=True)
all_bundles = {}
for i, turbine in enumerate(turbines, start=1):
    all_bundles[turbine] = build_turbine_bundle(
        turbine,
        binned,
        layout,
        FLEET,
        RELATIVE,
        MODEL,
    )
    print(f'  bundle {i}/{len(turbines)}: {turbine}', flush=True)

print('Applying site common-mode veto...', flush=True)
all_bundles = apply_site_common_mode_veto(
    all_bundles,
    MODEL,
)
print(f'Bundles ready. Elapsed: {time.perf_counter() - bundle_t0:.1f}s', flush=True)

bundles = {
    turbine: all_bundles[turbine]
    for turbine in TRAIN + TARGETS
}

MODEL_BUILD_SECONDS = time.perf_counter() - t0

# Pair reliability summary.
pair_rows = []
for turbine, bundle in bundles.items():
    table = bundle["pair_diagnostics"].reset_index(names="neighbour")
    table.insert(0, "turbine", turbine)
    pair_rows.append(
        table[
            [
                "turbine",
                "neighbour",
                "coverage",
                "pair_mad",
                "reliability_weight",
                "usable",
            ]
        ]
    )

pair_diagnostics = pd.concat(pair_rows, ignore_index=True)
display(pair_diagnostics.round(3))

# Keep the full event table in memory, but show only accepted events and
# important vetoes in the notebook.
event_rows = []
for turbine, bundle in bundles.items():
    table = bundle["boundaries"]
    if table is None or table.empty:
        continue

    out = table.reset_index(names="date")
    out.insert(0, "turbine", turbine)
    event_rows.append(out)

event_diagnostics = (
    pd.concat(event_rows, ignore_index=True)
    if event_rows
    else pd.DataFrame()
)

if not event_diagnostics.empty:
    important_reasons = {
        "yaw",
        "pelt_yaw",
        "sensor",
        "sensor_shadow",
        "site_common_mode",
        "low_confidence_prior",
    }

    event_view = event_diagnostics[
        event_diagnostics["accepted"]
        | event_diagnostics["reason"].isin(important_reasons)
    ][
        [
            "turbine",
            "date",
            "source",
            "change",
            "z",
            "pair_agreement",
            "event_confidence",
            "shrinkage",
            "reason",
            "accepted",
        ]
    ].copy()

    numeric = event_view.select_dtypes(include=[np.number]).columns
    event_view[numeric] = event_view[numeric].round(3)
    display(event_view.sort_values(["turbine", "date"]))

print(f"Relative-heading model build: {MODEL_BUILD_SECONDS:.1f}s")

Building relative-heading bundles for 16 turbines...
  bundle 1/16: PPP_WTG07
  bundle 2/16: PPP_WTG08
  bundle 3/16: PPP_WTG11
  bundle 4/16: PPP_WTG12
  bundle 5/16: PPP_WTG13
  bundle 6/16: PPP_WTG14
  bundle 7/16: PPP_WTG15
  bundle 8/16: PPP_WTG16
  bundle 9/16: PPP_WTG17
  bundle 10/16: PPP_WTG18
  bundle 11/16: PPP_WTG33
  bundle 12/16: SSS_WTG04
  bundle 13/16: SSS_WTG05
  bundle 14/16: SSS_WTG06
  bundle 15/16: SSS_WTG07
  bundle 16/16: SSS_WTG16
Applying site common-mode veto...
Bundles ready. Elapsed: 27.9s


,turbine,neighbour,coverage,pair_mad,reliability_weight,usable
0,PPP_WTG12,PPP_WTG11,0.625,3.555,0.256,True
1,PPP_WTG12,PPP_WTG13,0.653,6.055,0.129,True
2,PPP_WTG12,PPP_WTG14,0.614,8.402,0.043,True
3,PPP_WTG13,PPP_WTG14,0.817,5.033,0.275,True
4,PPP_WTG13,PPP_WTG12,0.707,6.192,0.124,True
5,PPP_WTG13,PPP_WTG11,0.832,7.804,0.058,True
6,PPP_WTG13,PPP_WTG08,0.843,7.669,0.039,True
7,PPP_WTG14,PPP_WTG13,0.774,5.210,0.253,True
8,PPP_WTG14,PPP_WTG12,0.608,8.662,0.035,True
9,PPP_WTG14,PPP_WTG08,0.768,6.125,0.059,True


,turbine,date,source,change,z,pair_agreement,event_confidence,shrinkage,reason,accepted
6,PPP_WTG12,2023-07-09,pelt,-33.198,23.921,0.000,0.000,0.000,sensor,False
7,PPP_WTG12,2023-07-17,rolling,-32.451,34.241,0.750,0.263,0.000,sensor,False
8,PPP_WTG12,2023-08-07,rolling,-8.654,8.388,0.900,0.000,0.000,sensor_shadow,False
9,PPP_WTG12,2023-08-08,pelt,-8.396,4.798,0.900,0.000,0.000,sensor_shadow,False
13,PPP_WTG12,2024-03-11,rolling,-6.030,3.093,0.699,0.000,0.000,site_common_mode,False
16,PPP_WTG13,2023-02-19,rolling,-3.614,7.227,0.751,0.321,0.321,yaw,True
20,PPP_WTG13,2023-07-27,pelt,-3.603,2.236,0.884,0.436,0.436,pelt_yaw,True
23,PPP_WTG13,2024-03-10,rolling,-6.261,5.067,0.671,0.000,0.000,site_common_mode,False
31,PPP_WTG14,2023-06-05,rolling,-3.435,3.259,0.680,0.142,0.000,low_confidence_prior,False
34,PPP_WTG14,2024-03-11,rolling,6.584,3.226,0.839,0.000,0.000,site_common_mode,False


Relative-heading model build: 27.9s


## 4. Strict turbine-level LOTO validation

Each fold:

1. holds out one labelled turbine;
2. estimates \(C\) and any eligible global \($\beta$\) using only the other labelled turbines;
3. uses the holdout turbine only through its unlabeled SCADA-derived prior/state features;
4. compares the state-aware prediction with the same-fold constant prior.

Because only a few reliable labelled state transitions survive, the model deliberately falls back to the physical prior \($\beta=1$\) instead of fitting an unstable slope.

The notebook reports MAE/RMSE. Legacy micro-step boundary metrics returned by the helper are intentionally not used for model selection.

In [4]:
t0 = time.perf_counter()
rows = []

for holdout in TRAIN:
    train_ids = [
        turbine
        for turbine in TRAIN
        if turbine != holdout
    ]

    C, beta = fit_global_beta(
        train_ids,
        bundles,
        labels,
        theta_star,
        MODEL,
    )

    prediction = predict_from_bundle(
        holdout,
        bundles[holdout],
        C,
        beta,
        theta_star,
    )

    metrics = score_prediction(
        prediction,
        labels[holdout],
        C - theta_star[holdout],
    )

    boundary_table = bundles[holdout]["boundaries"]
    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    rows.append(
        {
            "holdout": holdout,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "constant_mae": metrics["constant_mae"],
            "constant_rmse": metrics["constant_rmse"],
            "beta": beta,
            "n_states": int(
                bundles[holdout]["observables"]["cluster"].nunique()
            ),
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
        }
    )

LOTO_SECONDS = time.perf_counter() - t0
loto = pd.DataFrame(rows)

display(loto.round(3))

macro = pd.DataFrame(
    {
        "state_aware": [
            loto["mae"].mean(),
            loto["rmse"].mean(),
        ],
        "constant_prior": [
            loto["constant_mae"].mean(),
            loto["constant_rmse"].mean(),
        ],
    },
    index=["MAE", "RMSE"],
)

display(macro.round(3))

mae_gain = 1.0 - macro.loc["MAE", "state_aware"] / macro.loc["MAE", "constant_prior"]
rmse_gain = 1.0 - macro.loc["RMSE", "state_aware"] / macro.loc["RMSE", "constant_prior"]

print(
    f"Macro improvement: MAE {mae_gain:.1%}, RMSE {rmse_gain:.1%}"
)
print(f"LOTO scoring: {LOTO_SECONDS:.2f}s")

,holdout,mae,rmse,constant_mae,constant_rmse,beta,n_states,accepted_boundaries
0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none


,state_aware,constant_prior
MAE,0.390,0.522
RMSE,0.616,0.752


Macro improvement: MAE 25.3%, RMSE 18.1%
LOTO scoring: 0.05s


## 5. Frozen-boundary soft pair-quality ablation

This is a state-level robustness test, not a new detector. The accepted boundaries, \(B0\), \(\beta\), and calibration protocol are unchanged. Each state's daily relative-heading observations are aggregated with a continuous weight based on cross-pair spread, finite-pair coverage, and fixed historical pair reliability. No day is hard-deleted.

In [5]:
from yaw_model import apply_pair_quality_state_weights

if any(name not in globals() for name in ("bundles", "labels", "theta_star")):
    try:
        get_ipython().run_line_magic(
            "store",
            "-r bundles labels theta_star MODEL_CONFIG",
        )
    except Exception as exc:
        raise RuntimeError(
            "Run the data/build cells first, or restore the stored model state."
        ) from exc
if "MODEL" not in globals():
    MODEL = globals().get("MODEL_CONFIG")
if MODEL is None:
    raise RuntimeError("MODEL / MODEL_CONFIG is not available.")

SOFT_SPREAD_SCALE_DEG = 2.5
soft_bundles = apply_pair_quality_state_weights(
    bundles,
    MODEL,
    spread_scale_deg=SOFT_SPREAD_SCALE_DEG,
)

from yaw_model import run_loto

stage_tables = {}
stage_macros = {}
for stage_name, stage_bundles in {
    "all_days": bundles,
    "soft_pair_quality": soft_bundles,
}.items():
    table, macro_stage = run_loto(
        TRAIN, stage_bundles, labels, theta_star, MODEL
    )
    table.insert(0, "stage", stage_name)
    stage_tables[stage_name] = table
    stage_macros[stage_name] = macro_stage

soft_loto = pd.concat(stage_tables.values(), ignore_index=True)
display(soft_loto.round(3))
display(
    soft_loto.groupby("stage")[["mae", "rmse", "constant_mae", "constant_rmse"]]
    .mean().round(3)
)

quality_rows = []
for tid, bundle in soft_bundles.items():
    q = bundle["observables"]["state_quality_weight"]
    spread = bundle["observables"]["state_pair_spread_deg"]
    quality_rows.append({
        "turbine": tid,
        "median_day_quality": q.replace(0.0, np.nan).median(),
        "median_pair_spread_deg": spread.median(),
        "usable_quality_days": int((q > 0).sum()),
    })
display(pd.DataFrame(quality_rows).round(3))
print("Boundaries and calibration are frozen; only state-level day weights changed.")

,stage,holdout,mae,rmse,constant_mae,constant_rmse,beta,states,boundaries
0,all_days,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,all_days,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,all_days,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
3,soft_pair_quality,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
4,soft_pair_quality,PPP_WTG13,0.578,0.958,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
5,soft_pair_quality,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none


,mae,rmse,constant_mae,constant_rmse
stage,,,,
all_days,0.39,0.616,0.522,0.752
soft_pair_quality,0.38,0.609,0.522,0.752


,turbine,median_day_quality,median_pair_spread_deg,usable_quality_days
0,PPP_WTG12,0.741,1.084,569
1,PPP_WTG13,0.617,1.743,716
2,PPP_WTG14,0.618,1.748,662
3,PPP_WTG17,0.794,0.744,710
4,SSS_WTG06,0.864,0.904,607


Boundaries and calibration are frozen; only state-level day weights changed.


## 6. Pair-quality weighting robustness and attribution

The scale is audited at three pre-declared values. This is not a parameter search: boundaries remain frozen, and every scale uses the same turbine-level LOTO protocol. Pair residual levels have an arbitrary offset, so the WTG13 attribution compares only duration-centred dynamic shape (with the heading sign reversed), not raw residual levels against absolute yaw labels. Target rows are sensitivity only.

In [6]:
from fleet_context import circular_mean_deg
from yaw_model import (
    apply_pair_quality_state_weights,
    weighted_circular_median_deg,
)

if any(name not in globals() for name in ("bundles", "labels", "theta_star")):
    try:
        get_ipython().run_line_magic(
            "store",
            "-r bundles labels theta_star MODEL_CONFIG",
        )
    except Exception as exc:
        raise RuntimeError(
            "Run the data/build cells first, or restore the stored model state."
        ) from exc
if "MODEL" not in globals():
    MODEL = globals().get("MODEL_CONFIG")
if MODEL is None:
    raise RuntimeError("MODEL / MODEL_CONFIG is not available.")
if "soft_bundles" not in globals():
    soft_bundles = apply_pair_quality_state_weights(
        bundles,
        MODEL,
        spread_scale_deg=2.5,
    )

WEIGHTING_SCALES = {
    "conservative": 4.0,
    "current": 2.5,
    "aggressive": 1.5,
}
robust_tables = {}
robust_bundles = {}
robust_rows = []

baseline_table, _ = run_loto(
    TRAIN, bundles, labels, theta_star, MODEL
)
baseline_table.insert(0, "weighting", "all_days")
baseline_table.insert(1, "spread_scale_deg", np.nan)
robust_rows.append(baseline_table)

for weighting_name, scale in WEIGHTING_SCALES.items():
    stage = apply_pair_quality_state_weights(
        bundles,
        MODEL,
        spread_scale_deg=scale,
    )
    robust_bundles[weighting_name] = stage
    table, _ = run_loto(TRAIN, stage, labels, theta_star, MODEL)
    table.insert(0, "weighting", weighting_name)
    table.insert(1, "spread_scale_deg", scale)
    robust_tables[weighting_name] = table
    robust_rows.append(table)

robust_loto = pd.concat(robust_rows, ignore_index=True)
display(robust_loto.round(3))
display(
    robust_loto.groupby("weighting")[["mae", "rmse", "constant_mae", "constant_rmse"]]
    .mean().round(3)
)

def state_level_attribution(bundle, turbine, label=None):
    obs = bundle[turbine]["observables"]
    rows = []
    label_daily = None
    if label is not None:
        label_daily = label.groupby(label.index.normalize()).median()
    for state, part in obs.groupby("cluster", observed=True):
        valid = part["relative_heading_smooth"].notna()
        values = part.loc[valid, "relative_heading_smooth"].to_numpy(float)
        weights = part.loc[valid, "state_quality_weight"].to_numpy(float)
        level_all = circular_median_deg(values)
        level_soft = weighted_circular_median_deg(values, weights)
        label_level = np.nan
        if label_daily is not None:
            label_level = label_daily.reindex(part.index).dropna().median()
        rows.append({
            "turbine": turbine,
            "state": int(state),
            "days": int(valid.sum()),
            "median_pair_spread_deg": part["state_pair_spread_deg"].median(),
            "effective_days": (weights.sum() ** 2 / (weights ** 2).sum()) if (weights > 0).any() else 0.0,
            "relative_level_all_deg": level_all,
            "relative_level_soft_deg": level_soft,
            "soft_minus_all_deg": wrap_180(level_soft - level_all),
            "label_level_deg": label_level,
        })
    table = pd.DataFrame(rows)
    if table.empty:
        return table

    # The relative-heading residual has an arbitrary pair-reference offset.
    # Compare only its dynamic shape with the labelled yaw states: negate the
    # heading residual and centre both trajectories with duration weights.
    duration = table["days"].to_numpy(float)
    duration = duration / duration.sum()
    yaw_all = -table["relative_level_all_deg"].to_numpy(float)
    yaw_soft = -table["relative_level_soft_deg"].to_numpy(float)
    all_centre = circular_mean_deg(yaw_all, duration)
    soft_centre = circular_mean_deg(yaw_soft, duration)
    table["heading_yaw_all_centered_deg"] = [wrap_180(x - all_centre) for x in yaw_all]
    table["heading_yaw_soft_centered_deg"] = [wrap_180(x - soft_centre) for x in yaw_soft]
    labels_arr = table["label_level_deg"].to_numpy(float)
    label_ok = np.isfinite(labels_arr)
    if label_ok.any():
        label_w = duration[label_ok] / duration[label_ok].sum()
        label_centre = circular_mean_deg(labels_arr[label_ok], label_w)
        centred_labels = np.full(len(table), np.nan)
        centred_labels[label_ok] = [wrap_180(x - label_centre) for x in labels_arr[label_ok]]
        table["label_centered_deg"] = centred_labels
        table["all_shape_error_deg"] = [
            wrap_180(a - b) if np.isfinite(b) else np.nan
            for a, b in zip(table["heading_yaw_all_centered_deg"], centred_labels)
        ]
        table["soft_shape_error_deg"] = [
            wrap_180(a - b) if np.isfinite(b) else np.nan
            for a, b in zip(table["heading_yaw_soft_centered_deg"], centred_labels)
        ]
    return table

wtg13_attr = state_level_attribution(
    robust_bundles["current"],
    "PPP_WTG13",
    labels["PPP_WTG13"],
)
print("WTG13 state-level attribution: current soft weighting")
display(wtg13_attr.round(3))

C_ref, beta_ref = fit_global_beta(
    TRAIN, bundles, labels, theta_star, MODEL
)
target_rows = []
for turbine in TARGETS:
    for stage_name, stage_bundle in {
        "all_days": bundles,
        "soft_current": robust_bundles["current"],
    }.items():
        pred = predict_from_bundle(
            turbine, stage_bundle[turbine], C_ref, beta_ref, theta_star
        )
        state_values = pred.groupby("cluster")["prediction"].median()
        target_rows.append({
            "turbine": turbine,
            "stage": stage_name,
            "beta": beta_ref,
            "states": int(pred["cluster"].nunique()),
            "state_levels_deg": ", ".join(f"{v:.3f}" for v in state_values),
            "duration_mean_deg": pred["prediction"].mean(),
            "prediction_min": pred["prediction"].min(),
            "prediction_max": pred["prediction"].max(),
        })

print("Blind-target sensitivity: same C and beta, all-days versus current soft weighting")
display(pd.DataFrame(target_rows).round(3))
print("Boundaries, C, beta, and lambda were not re-tuned in this audit.")

,weighting,spread_scale_deg,holdout,mae,rmse,constant_mae,constant_rmse,beta,states,boundaries
0,all_days,NaN,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
1,all_days,NaN,PPP_WTG13,0.608,0.979,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
2,all_days,NaN,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
3,conservative,4.0,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
4,conservative,4.0,PPP_WTG13,0.595,0.968,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
5,conservative,4.0,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
6,current,2.5,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none
7,current,2.5,PPP_WTG13,0.578,0.958,1.004,1.387,1.0,3,"2023-02-19, 2023-07-27"
8,current,2.5,PPP_WTG14,0.206,0.262,0.206,0.262,1.0,1,none
9,aggressive,1.5,PPP_WTG12,0.357,0.608,0.357,0.608,1.0,1,none


,mae,rmse,constant_mae,constant_rmse
weighting,,,,
aggressive,0.384,0.610,0.522,0.752
all_days,0.390,0.616,0.522,0.752
conservative,0.386,0.613,0.522,0.752
current,0.380,0.609,0.522,0.752


WTG13 state-level attribution: current soft weighting


,turbine,state,days,median_pair_spread_deg,effective_days,relative_level_all_deg,relative_level_soft_deg,soft_minus_all_deg,label_level_deg,heading_yaw_all_centered_deg,heading_yaw_soft_centered_deg,label_centered_deg,all_shape_error_deg,soft_shape_error_deg
0,PPP_WTG13,0,49,3.972,31.170,3.970,4.111,0.141,-9.666,-3.930,-3.986,-3.034,-0.896,-0.952
1,PPP_WTG13,1,158,2.619,119.953,1.707,2.019,0.312,-8.030,-1.667,-1.894,-1.398,-0.269,-0.496
2,PPP_WTG13,2,524,1.491,445.782,-0.830,-0.819,0.011,-5.927,0.870,0.944,0.705,0.165,0.238


Blind-target sensitivity: same C and beta, all-days versus current soft weighting


,turbine,stage,beta,states,state_levels_deg,duration_mean_deg,prediction_min,prediction_max
0,PPP_WTG17,all_days,1.0,4,"-5.246, -5.732, -6.680, -1.422",-3.602,-6.680,-1.422
1,PPP_WTG17,soft_current,1.0,4,"-5.280, -5.652, -6.635, -1.441",-3.602,-6.635,-1.441
2,SSS_WTG06,all_days,1.0,4,"-4.079, -8.162, -3.826, -3.729",-4.572,-8.162,-3.729
3,SSS_WTG06,soft_current,1.0,4,"-4.090, -7.923, -3.859, -3.910",-4.572,-7.923,-3.859


Boundaries, C, beta, and lambda were not re-tuned in this audit.


## 7. Interaction audit: state weighting × amplitude

This small audit keeps the accepted boundaries and $B0=C-\theta^\star$ fixed. It compares all-days versus soft pair-quality state levels at $\lambda\in\{0,0.75,1\}$ with fixed $\beta=1$. The three values mean stable amplitude, partial amplitude, and full amplitude; this is an interaction check, not a new parameter search.

In [7]:
from yaw_model import (
    apply_full_amplitude_stable_states,
    blend_stable_amplitude,
)

if "soft_bundles" not in globals():
    soft_bundles = apply_pair_quality_state_weights(
        bundles, MODEL, spread_scale_deg=2.5
    )

full_bundles = apply_full_amplitude_stable_states(bundles, MODEL)
soft_full_bundles = apply_full_amplitude_stable_states(soft_bundles, MODEL)

def run_fixed_beta_loto(stage_bundles):
    rows = []
    for holdout in TRAIN:
        fit_ids = [tid for tid in TRAIN if tid != holdout]
        C_fixed = float(np.mean([
            labels[tid].mean() + theta_star[tid]
            for tid in fit_ids
        ]))
        pred = predict_from_bundle(
            holdout, stage_bundles[holdout], C_fixed, 1.0, theta_star
        )
        metrics = score_prediction(
            pred, labels[holdout], C_fixed - theta_star[holdout]
        )
        boundaries = stage_bundles[holdout]["boundaries"]
        active_col = (
            "prediction_active"
            if "prediction_active" in boundaries.columns
            else "accepted"
        )
        active = boundaries[active_col].fillna(False) if len(boundaries) else pd.Series(dtype=bool)
        accepted = boundaries.index[active] if len(boundaries) else []
        rows.append({
            "holdout": holdout,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "constant_mae": metrics["constant_mae"],
            "constant_rmse": metrics["constant_rmse"],
            "states": metrics["n_states"],
            "boundaries": ", ".join(
                pd.Timestamp(d).date().isoformat() for d in accepted
            ) or "none",
        })
    return pd.DataFrame(rows)

interaction_rows = []
for estimator, stable_stage, full_stage in (
    ("all_days", bundles, full_bundles),
    ("soft_2.5deg", soft_bundles, soft_full_bundles),
):
    for amplitude_lambda in (0.0, 0.75, 1.0):
        mixed_stage = blend_stable_amplitude(
            stable_stage, full_stage, amplitude_lambda
        )
        table = run_fixed_beta_loto(mixed_stage)
        table.insert(0, "estimator", estimator)
        table.insert(1, "lambda_amp", amplitude_lambda)
        interaction_rows.append(table)

interaction_loto = pd.concat(interaction_rows, ignore_index=True)
print("Interaction audit: fixed boundaries, B0, and beta=1")
display(interaction_loto.round(3))
display(
    interaction_loto.groupby(["estimator", "lambda_amp"])[
        ["mae", "rmse", "constant_mae", "constant_rmse"]
    ].mean().round(3)
)

Interaction audit: fixed boundaries, B0, and beta=1


,estimator,lambda_amp,holdout,mae,rmse,constant_mae,constant_rmse,states,boundaries
0,all_days,0.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
1,all_days,0.00,PPP_WTG13,0.608,0.979,1.004,1.387,3,"2023-02-19, 2023-07-27"
2,all_days,0.00,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
3,all_days,0.75,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
4,all_days,0.75,PPP_WTG13,0.509,0.702,1.004,1.387,3,"2023-02-19, 2023-07-27"
5,all_days,0.75,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
6,all_days,1.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none
7,all_days,1.00,PPP_WTG13,0.584,0.755,1.004,1.387,3,"2023-02-19, 2023-07-27"
8,all_days,1.00,PPP_WTG14,0.206,0.262,0.206,0.262,1,none
9,soft_2.5deg,0.00,PPP_WTG12,0.357,0.608,0.357,0.608,1,none


mae   rmse  constant_mae  constant_rmse
estimator   lambda_amp                                           
all_days    0.00        0.390  0.616         0.522          0.752
            0.75        0.357  0.524         0.522          0.752
            1.00        0.382  0.541         0.522          0.752
soft_2.5deg 0.00        0.380  0.609         0.522          0.752
            0.75        0.358  0.524         0.522          0.752
            1.00        0.382  0.541         0.522          0.752

## 8. Optional validation export

This cell prepares the selected validation candidate without writing a file by default. Set $\texttt{EXPORT\_PATH}$ explicitly before running it. The default candidate is soft pair-quality state levels with $\lambda=0.75$; change only $\texttt{EXPORT\_ESTIMATOR}$ to $\texttt{all\_days}$ for the tied all-days ablation.

In [8]:
from yaw_model import (
    apply_full_amplitude_stable_states,
    blend_stable_amplitude,
)

EXPORT_ESTIMATOR = "soft_2.5deg"  # or "all_days"
EXPORT_ANCHOR = "corrected_mean"  # or "legacy_mean"
# Match the candidate's direct farm-first / global-fallback rule.
EXPORT_SHRINK_KAPPA = 0.0
EXPORT_LAMBDA = 0.75
EXPORT_TARGET = "PPP_WTG17"
EXPORT_PATH = ROOT / "submissions" / "Results_33_T3_5.csv"
#None  # set e.g. ROOT / "submissions" / "Results_33_T3_validation_soft075.csv"

if EXPORT_ESTIMATOR == "soft_2.5deg":
    stable_stage = soft_bundles
else:
    stable_stage = bundles
full_stage = apply_full_amplitude_stable_states(stable_stage, MODEL)
candidate_stage = blend_stable_amplitude(
    stable_stage, full_stage, EXPORT_LAMBDA
)
export_farm_map = {
    turbine: turbine_farm_key(turbine)
    for turbine in turbines
}
if EXPORT_ANCHOR == "corrected_mean":
    export_anchor_fit = fit_farm_anchor(
        TRAIN,
        candidate_stage,
        labels,
        theta_star,
        farm_map=export_farm_map,
        method="mean",
        shrink_kappa=EXPORT_SHRINK_KAPPA,
    )
    C_corrected = export_anchor_fit.center_for(
        EXPORT_TARGET,
        farm_map=export_farm_map,
    )
    C_export = C_corrected
elif EXPORT_ANCHOR == "legacy_mean":
    C_corrected = np.nan
    C_export = legacy_global_anchor(TRAIN, labels, theta_star)
else:
    raise ValueError("EXPORT_ANCHOR must be corrected_mean or legacy_mean")
pred_export = predict_from_bundle(
    EXPORT_TARGET, candidate_stage[EXPORT_TARGET], C_export, 1.0, theta_star
)
dates_export = pd.date_range("2023-01-01", "2024-12-31", freq="D")
submission_predictions = pd.DataFrame(index=dates_export)
submission_predictions["yaw_misalignment_deg"] = pred_export["prediction"].reindex(dates_export)
submission_predictions["cluster"] = pred_export["cluster"].reindex(dates_export)
submission_predictions["yaw_misalignment_deg"] = submission_predictions["yaw_misalignment_deg"].fillna(C_export - theta_star[EXPORT_TARGET])
submission_predictions["cluster"] = submission_predictions["cluster"].fillna(0).astype(int)
submission_predictions = submission_predictions.reset_index(names="date")
submission_predictions["turbine_id"] = EXPORT_TARGET
submission_predictions["date"] = submission_predictions["date"].dt.strftime("%Y-%m-%d")
submission_predictions = submission_predictions[["turbine_id", "date", "yaw_misalignment_deg", "cluster"]]

# All submission checks use only the explicit candidate table.
assert submission_predictions["turbine_id"].eq(EXPORT_TARGET).all()
assert len(submission_predictions) == 731
assert submission_predictions["date"].nunique() == 731
assert submission_predictions["date"].is_monotonic_increasing
assert submission_predictions["yaw_misalignment_deg"].notna().all()
assert np.isfinite(submission_predictions["yaw_misalignment_deg"]).all()
assert submission_predictions["cluster"].notna().all()
if EXPORT_ANCHOR == "corrected_mean":
    assert np.isclose(C_corrected, -6.155553, atol=1e-6)
    if EXPORT_TARGET == "PPP_WTG17":
        assert np.isclose(C_export - theta_star[EXPORT_TARGET], -3.672, atol=0.01)

print(f"candidate={EXPORT_ESTIMATOR}, anchor={EXPORT_ANCHOR}, lambda={EXPORT_LAMBDA}, beta=1, C={C_export:.6f}")
print(f"rows={len(submission_predictions)}, range=({submission_predictions.yaw_misalignment_deg.min():.3f}, {submission_predictions.yaw_misalignment_deg.max():.3f})")
if EXPORT_PATH is None:
    print("No CSV written. Set EXPORT_PATH explicitly to export.")
else:
    EXPORT_PATH = Path(EXPORT_PATH)
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission_predictions.to_csv(EXPORT_PATH, index=False, float_format="%.6f")
    print(f"Wrote {len(submission_predictions)} rows to {EXPORT_PATH}")

candidate=soft_2.5deg, anchor=corrected_mean, lambda=0.75, beta=1, C=-6.155553
rows=731, range=(-8.015, 0.191)
Wrote 731 rows to E:\EnergyHacks\github_release\submissions\Results_33_T3_5.csv


## 9. Final fit and unlabeled target diagnostics

The final calibration uses all three labelled training turbines.

Target diagnostics below are **not validation scores**. They are shown to make deployment behaviour auditable:

- accepted state dates;
- detector source and confidence;
- correction range around the constant prior;
- resulting prediction range.

This is especially important because the targets may contain larger unlabeled state structure than the training turbines.

In [9]:
RELEASE_C, RELEASE_BETA = fit_global_beta(
    TRAIN,
    bundles,
    labels,
    theta_star,
    MODEL,
)

release_predictions = {
    turbine: predict_from_bundle(
        turbine,
        bundles[turbine],
        RELEASE_C,
        RELEASE_BETA,
        theta_star,
    )
    for turbine in TRAIN + TARGETS
}

diagnostic_rows = []

for turbine in TRAIN + TARGETS:
    bundle = bundles[turbine]
    boundary_table = bundle["boundaries"]

    accepted = (
        boundary_table.loc[
            boundary_table["accepted"].fillna(False)
        ]
        if len(boundary_table)
        else pd.DataFrame()
    )

    correction = bundle["observables"][
        "relative_prior_correction"
    ].fillna(0.0)

    pred = release_predictions[turbine]["prediction"]

    diagnostic_rows.append(
        {
            "turbine": turbine,
            "constant_prior": RELEASE_C - theta_star[turbine],
            "accepted_boundaries": ", ".join(
                pd.Timestamp(date).date().isoformat()
                for date in accepted.index
            )
            or "none",
            "accepted_sources": ", ".join(
                accepted["source"].astype(str).tolist()
            )
            if len(accepted)
            else "none",
            "mean_event_confidence": (
                float(accepted["event_confidence"].mean())
                if len(accepted)
                else 0.0
            ),
            "correction_range": float(
                correction.max() - correction.min()
            ),
            "prediction_min": float(pred.min()),
            "prediction_max": float(pred.max()),
        }
    )

deployment_diagnostics = pd.DataFrame(diagnostic_rows)

display(deployment_diagnostics.round(3))

print(f"Release baseline C = {RELEASE_C:.3f}")
print(f"Release baseline beta = {RELEASE_BETA:.3f}")
print(
    {
        "load_and_B0_seconds": round(LOAD_SECONDS, 2),
        "relative_model_seconds": round(MODEL_BUILD_SECONDS, 2),
        "LOTO_seconds": round(LOTO_SECONDS, 2),
    }
)

# Basic release sanity checks.
for turbine, pred in release_predictions.items():
    values = pred["prediction"].to_numpy(dtype=float)
    assert np.isfinite(values).all(), f"Non-finite prediction for {turbine}"
    assert np.max(np.abs(values)) <= 90.0, f"Prediction out of bounds for {turbine}"

print("Release sanity checks passed.")

,turbine,constant_prior,accepted_boundaries,accepted_sources,mean_event_confidence,correction_range,prediction_min,prediction_max
0,PPP_WTG12,-2.642,none,none,0.000,0.000,-2.642,-2.642
1,PPP_WTG13,-6.437,"2023-02-19, 2023-07-27","rolling, pelt",0.379,1.631,-7.764,-6.133
2,PPP_WTG14,-1.731,none,none,0.000,0.000,-1.731,-1.731
3,PPP_WTG17,-3.602,"2023-06-25, 2023-10-15, 2024-01-07","rolling, rolling, rolling",0.512,5.258,-6.680,-1.422
4,SSS_WTG06,-4.572,"2023-05-28, 2023-09-24, 2024-10-13","rolling, rolling, rolling",0.542,4.434,-8.162,-3.729


Release baseline C = -6.085
Release baseline beta = 1.000
{'load_and_B0_seconds': 173.35, 'relative_model_seconds': 27.92, 'LOTO_seconds': 0.05}
Release sanity checks passed.


## 10. FarmAnchor exploration: state-consistent robust absolute centre

The release model estimates one shared centre from long-run turbine means:

$$C = \mathrm{mean}_i(\bar y_i + \theta_i^*)$$

This section leaves the frozen relative-heading dynamics unchanged and only tests alternative estimates of the absolute centre. For a labelled turbine, the daily anchor observation is

$$a_{i,d}=y_i(d)+c_{i,0.75}^{\mathrm{soft}}(d)+\theta_i^*.$$

The exploration first forms daily state-consistent anchor observations, then reduces each labelled turbine to a quality-weighted turbine-level arithmetic mean `A_i`. The shared centre is a robust Huber centre across those turbine means. If same-farm labelled support exists, its centre is preferred; otherwise the global centre is used. The default farm key uses the identifier prefix (`PPP` / `SSS`); replace `FARM_MAP` with metadata-derived site labels when available.

The current full-fit PPP result is `C_corrected = -6.155553°`. Because the development labels all come from PPP, this notebook currently tests a robust corrected global/farm anchor rather than demonstrating cross-farm transfer.

Important: a farm-specific `C_f` requires an absolute labelled or calibration reference. Relative headings alone cannot identify a common rotation of all turbines.

In [10]:
from yaw_farm_anchor import (
    fit_farm_anchor,
    legacy_global_anchor,
    run_anchor_loto,
    turbine_farm_key,
)
from yaw_model import (
    apply_full_amplitude_stable_states,
    apply_pair_quality_state_weights,
    blend_stable_amplitude,
)

# Keep the release candidate fixed: soft pair weighting and lambda=0.75.
ANCHOR_SPREAD_SCALE_DEG = 2.5
ANCHOR_LAMBDA = 0.75
# Direct farm-first selection for the candidate; positive shrinkage is
# retained only as an optional diagnostic in yaw_farm_anchor.py.
ANCHOR_SHRINK_KAPPA = 0.0

if 'soft_bundles' not in globals():
    soft_bundles = apply_pair_quality_state_weights(
        bundles, MODEL, spread_scale_deg=ANCHOR_SPREAD_SCALE_DEG
    )
anchor_full_bundles = apply_full_amplitude_stable_states(
    soft_bundles, MODEL
)
anchor_stage = blend_stable_amplitude(
    soft_bundles, anchor_full_bundles, ANCHOR_LAMBDA
)

# The prefix is only a fallback grouping rule. Supply a metadata mapping
# here if the layout contains a more authoritative farm/site field.
FARM_MAP = {turbine: turbine_farm_key(turbine) for turbine in turbines}
print('Anchor stage: soft pair weighting + lambda=0.75')
print('Farm groups:', pd.Series(FARM_MAP).value_counts().to_dict())

Anchor stage: soft pair weighting + lambda=0.75
Farm groups: {'PPP': 11, 'SSS': 5}


In [11]:
# Strict turbine-LOTO: only the non-held-out labelled turbines estimate C.
anchor_loto = run_anchor_loto(
    TRAIN,
    anchor_stage,
    labels,
    theta_star,
    farm_map=FARM_MAP,
    shrink_kappa=ANCHOR_SHRINK_KAPPA,
    beta=1.0,
)

display(anchor_loto.round(3))
anchor_macro = (
    anchor_loto.groupby('anchor')[
        ['mae', 'rmse', 'constant_mae', 'constant_rmse']
    ]
    .mean()
)
display(anchor_macro.round(3))

legacy_macro = anchor_macro.loc['legacy_mean']
for name in ['corrected_mean', 'farm_median', 'farm_huber']:
    mae_gain = 1.0 - anchor_macro.loc[name, 'mae'] / legacy_macro['mae']
    rmse_gain = 1.0 - anchor_macro.loc[name, 'rmse'] / legacy_macro['rmse']
    print(f'{name}: MAE change={mae_gain:+.1%}, RMSE change={rmse_gain:+.1%}')

if len(set(FARM_MAP[turbine] for turbine in TRAIN)) == 1:
    print('Note: all labelled training turbines are in one farm; cross-farm shrinkage is not identifiable in this LOTO.')

,anchor,holdout,farm,C,B0,mae,rmse,constant_mae,constant_rmse,states
0,legacy_mean,PPP_WTG12,PPP,-6.194,-2.750,0.357,0.608,0.357,0.608,1
1,corrected_mean,PPP_WTG12,PPP,-6.187,-2.744,0.357,0.604,0.357,0.604,1
2,farm_median,PPP_WTG12,PPP,-6.309,-2.866,0.457,0.677,0.457,0.677,1
3,farm_huber,PPP_WTG12,PPP,-6.260,-2.816,0.413,0.645,0.413,0.645,1
4,legacy_mean,PPP_WTG13,PPP,-5.992,-6.344,0.510,0.702,1.004,1.387,3
5,corrected_mean,PPP_WTG13,PPP,-6.115,-6.467,0.427,0.663,1.043,1.367,3
6,farm_median,PPP_WTG13,PPP,-6.210,-6.562,0.368,0.647,1.075,1.360,3
7,farm_huber,PPP_WTG13,PPP,-6.201,-6.553,0.374,0.648,1.072,1.360,3
8,legacy_mean,PPP_WTG14,PPP,-6.070,-1.715,0.206,0.262,0.206,0.262,1
9,corrected_mean,PPP_WTG14,PPP,-6.164,-1.809,0.175,0.262,0.175,0.262,1


,mae,rmse,constant_mae,constant_rmse
anchor,,,,
corrected_mean,0.320,0.510,0.525,0.744
farm_huber,0.327,0.528,0.560,0.765
farm_median,0.357,0.548,0.592,0.786
legacy_mean,0.358,0.524,0.522,0.752


corrected_mean: MAE change=+10.6%, RMSE change=+2.7%
farm_median: MAE change=+0.2%, RMSE change=-4.7%
farm_huber: MAE change=+8.5%, RMSE change=-0.8%
Note: all labelled training turbines are in one farm; cross-farm shrinkage is not identifiable in this LOTO.


In [12]:
# Full-fit diagnostics and blind-target anchor transfer.
anchor_fits = {
    method: fit_farm_anchor(
        TRAIN,
        anchor_stage,
        labels,
        theta_star,
        farm_map=FARM_MAP,
        method=method,
        shrink_kappa=ANCHOR_SHRINK_KAPPA,
    )
    for method in ['mean', 'median', 'huber']
}

print(f'Legacy C = {legacy_global_anchor(TRAIN, labels, theta_star):.6f}')
for method, fit in anchor_fits.items():
    print(f'{method}: global C = {fit.global_center:.6f}; farm C = {fit.farm_centers}')

print('Turbine-level robust anchor summaries:')
display(anchor_fits['median'].turbine_summary.round(3))

target_anchor_rows = []
legacy_c = legacy_global_anchor(TRAIN, labels, theta_star)
for turbine in TARGETS:
    target_anchor_rows.append({
        'turbine': turbine,
        'anchor': 'legacy_mean',
        'farm': FARM_MAP[turbine],
        'C': legacy_c,
        'B0': legacy_c - theta_star[turbine],
    })
    for method, fit in anchor_fits.items():
        centre = fit.center_for(turbine, farm_map=FARM_MAP)
        target_anchor_rows.append({
            'turbine': turbine,
            'anchor': f'farm_{method}',
            'farm': FARM_MAP[turbine],
            'C': centre,
            'B0': centre - theta_star[turbine],
        })

target_anchor_table = pd.DataFrame(target_anchor_rows)
display(target_anchor_table.round(3))
print('No target labels are used in the anchor transfer above.')

Legacy C = -6.085367
mean: global C = -6.155553; farm C = {'PPP': -6.1555529139667335}
median: global C = -6.209986; farm C = {'PPP': -6.209986378127094}
huber: global C = -6.220811; farm C = {'PPP': -6.220811478831876}
Turbine-level robust anchor summaries:


,turbine,n_days,n_eff_days,anchor_mean_deg,anchor_median_deg,anchor_mad_deg,farm,anchor_location_deg
0,PPP_WTG12,569,485.493,-6.092,-6.192,0.0,PPP,-6.192
1,PPP_WTG13,716,591.183,-6.236,-6.309,0.0,PPP,-6.309
2,PPP_WTG14,662,545.723,-6.139,-6.210,0.0,PPP,-6.210


,turbine,anchor,farm,C,B0
0,PPP_WTG17,legacy_mean,PPP,-6.085,-3.602
1,PPP_WTG17,farm_mean,PPP,-6.156,-3.672
2,PPP_WTG17,farm_median,PPP,-6.210,-3.727
3,PPP_WTG17,farm_huber,PPP,-6.221,-3.737
4,SSS_WTG06,legacy_mean,SSS,-6.085,-4.572
5,SSS_WTG06,farm_mean,SSS,-6.156,-4.643
6,SSS_WTG06,farm_median,SSS,-6.210,-4.697
7,SSS_WTG06,farm_huber,SSS,-6.221,-4.708


No target labels are used in the anchor transfer above.


### Candidate default: PARS FarmAnchor corrected centre

The LOTO audit indicates that the main gain comes from calibrating the absolute centre in the same coordinate system as the final state correction. The exploration default below uses the historical internal name `method='mean'`; in the implementation this route applies the robust Huber location to the quality-weighted turbine-level anchor means. The relative-heading stage, soft weighting, `lambda=0.75`, and `beta=1` remain fixed. This is a candidate path only; the release notebook remains unchanged.

In [13]:
DEFAULT_ANCHOR_NAME = 'corrected_mean'
# Historical name retained: method='mean' applies the robust Huber
# centre to the quality-weighted turbine-level anchor means.
DEFAULT_ANCHOR_METHOD = 'mean'
DEFAULT_ANCHOR_BETA = 1.0

DEFAULT_ANCHOR_FIT = fit_farm_anchor(
    TRAIN,
    anchor_stage,
    labels,
    theta_star,
    farm_map=FARM_MAP,
    method=DEFAULT_ANCHOR_METHOD,
    shrink_kappa=ANCHOR_SHRINK_KAPPA,
)

candidate_predictions = {}
candidate_rows = []
for turbine in TRAIN + TARGETS:
    centre = DEFAULT_ANCHOR_FIT.center_for(
        turbine,
        farm_map=FARM_MAP,
    )
    prediction = predict_from_bundle(
        turbine,
        anchor_stage[turbine],
        centre,
        DEFAULT_ANCHOR_BETA,
        theta_star,
    )
    candidate_predictions[turbine] = prediction
    row = {
        'turbine': turbine,
        'farm': FARM_MAP[turbine],
        'C': centre,
        'B0': centre - theta_star[turbine],
        'states': int(prediction['cluster'].nunique()),
        'prediction_min': float(prediction['prediction'].min()),
        'prediction_max': float(prediction['prediction'].max()),
        'prediction_mean': float(prediction['prediction'].mean()),
    }
    if turbine in labels:
        metrics = score_prediction(
            prediction,
            labels[turbine],
            centre - theta_star[turbine],
        )
        row['in_sample_mae'] = metrics['mae']
        row['in_sample_rmse'] = metrics['rmse']
    candidate_rows.append(row)

candidate_deployment = pd.DataFrame(candidate_rows)
print(
    f'Default candidate: {DEFAULT_ANCHOR_NAME}; C_global={DEFAULT_ANCHOR_FIT.global_center:.6f}; beta={DEFAULT_ANCHOR_BETA:.1f}'
)
print(f'Farm centres: {DEFAULT_ANCHOR_FIT.farm_centers}')
display(candidate_deployment.round(3))

Default candidate: corrected_mean; C_global=-6.155553; beta=1.0
Farm centres: {'PPP': -6.1555529139667335}


,turbine,farm,C,B0,states,prediction_min,prediction_max,prediction_mean,in_sample_mae,in_sample_rmse
0,PPP_WTG12,PPP,-6.156,-2.712,1,-2.712,-2.712,-2.712,0.366,0.588
1,PPP_WTG13,PPP,-6.156,-6.507,3,-9.790,-5.773,-6.507,0.401,0.655
2,PPP_WTG14,PPP,-6.156,-1.801,1,-1.801,-1.801,-1.801,0.178,0.260
3,PPP_WTG17,PPP,-6.156,-3.672,4,-8.015,0.191,-3.672,NaN,NaN
4,SSS_WTG06,SSS,-6.156,-4.643,4,-10.074,-3.288,-4.643,NaN,NaN


## 11. Anchor stability + theta_star stability audit

This section is diagnostic only. It reuses the same rolling power-vs-vane estimator and the same frozen relative-state observations as the candidate model. It does not change `candidate_predictions`, the default anchor, or the export path.

The audit asks two separate questions:

1. Does the turbine-specific `theta_star` move across quarters, or is the long-run median a reasonable reference?
2. After adding the observed label and frozen state correction, does the farm-level absolute anchor remain stable across quarters?

For the anchor table, quarterly turbine estimates are averaged with equal turbine weight, so a quarter with many observations from one turbine cannot dominate the farm center.

In [14]:
print('Building theta_star stability audit...', flush=True)
theta_quarter_rows = []
theta_stability_rows = []

for i, turbine in enumerate(TRAIN + TARGETS, start=1):
    series = theta_series[turbine].dropna().astype(float)
    assert np.isclose(theta_star[turbine], float(series.median()))
    grouped = series.groupby(series.index.to_period('Q'))
    for quarter, values in grouped:
        values = values.to_numpy(dtype=float)
        theta_quarter_rows.append({
            'turbine': turbine,
            'farm': FARM_MAP[turbine],
            'quarter': str(quarter),
            'n_valid_windows': int(len(values)),
            'theta_q_median_deg': float(np.median(values)),
            'theta_q_mean_deg': float(np.mean(values)),
            'theta_q_std_deg': float(np.std(values)),
            'theta_q_min_deg': float(np.min(values)),
            'theta_q_max_deg': float(np.max(values)),
        })
    elapsed_days = (series.index - series.index.min()).total_seconds() / 86400.0
    slope = float(np.polyfit(elapsed_days, series.to_numpy(dtype=float), 1)[0] * 365.25) if len(series) >= 2 else np.nan
    turbine_rows = [r for r in theta_quarter_rows if r['turbine'] == turbine]
    theta_stability_rows.append({
        'turbine': turbine,
        'farm': FARM_MAP[turbine],
        'theta_star_global_deg': float(theta_star[turbine]),
        'n_valid_windows': int(len(series)),
        'theta_daily_std_deg': float(series.std(ddof=0)),
        'theta_daily_range_deg': float(series.max() - series.min()),
        'theta_quarter_range_deg': float(max(r['theta_q_max_deg'] for r in turbine_rows) - min(r['theta_q_min_deg'] for r in turbine_rows)),
        'theta_drift_deg_per_year': slope,
    })
    print(f'  theta audit {i}/{len(TRAIN + TARGETS)}: {turbine}', flush=True)

theta_quarterly = (
    pd.DataFrame(theta_quarter_rows)
    .sort_values(['turbine', 'quarter'])
    .reset_index(drop=True)
)
theta_stability_summary = pd.DataFrame(theta_stability_rows).set_index('turbine')

display(theta_stability_summary.round(3))
display(theta_quarterly.round(3))

Building theta_star stability audit...
  theta audit 1/5: PPP_WTG12
  theta audit 2/5: PPP_WTG13
  theta audit 3/5: PPP_WTG14
  theta audit 4/5: PPP_WTG17
  theta audit 5/5: SSS_WTG06


,farm,theta_star_global_deg,n_valid_windows,theta_daily_std_deg,theta_daily_range_deg,theta_quarter_range_deg,theta_drift_deg_per_year
turbine,,,,,,,
PPP_WTG12,PPP,-3.444,698,2.214,9.863,9.863,0.280
PPP_WTG13,PPP,0.352,723,2.174,10.795,10.795,1.462
PPP_WTG14,PPP,-4.354,710,2.577,9.181,9.181,-0.297
PPP_WTG17,PPP,-2.483,659,2.455,8.858,8.858,1.532
SSS_WTG06,SSS,-1.513,731,3.339,11.151,11.151,0.971


,turbine,farm,quarter,n_valid_windows,theta_q_median_deg,theta_q_mean_deg,theta_q_std_deg,theta_q_min_deg,theta_q_max_deg
0,PPP_WTG12,PPP,2023Q1,78,-4.381,-3.758,1.252,-7.286,-0.576
1,PPP_WTG12,PPP,2023Q2,90,-3.416,-3.071,2.007,-6.442,1.363
2,PPP_WTG12,PPP,2023Q3,92,-2.527,-3.038,1.800,-6.492,1.457
3,PPP_WTG12,PPP,2023Q4,74,-4.412,-3.977,2.727,-6.439,2.577
4,PPP_WTG12,PPP,2024Q1,91,-3.483,-2.723,2.429,-5.453,1.486
5,PPP_WTG12,PPP,2024Q2,91,-3.417,-2.538,2.421,-6.308,2.360
6,PPP_WTG12,PPP,2024Q3,92,-3.480,-3.489,2.291,-6.411,1.428
7,PPP_WTG12,PPP,2024Q4,90,-3.439,-2.911,2.074,-6.381,0.429
8,PPP_WTG13,PPP,2023Q1,90,-2.523,-1.300,2.471,-5.396,3.508
9,PPP_WTG13,PPP,2023Q2,91,-3.429,-3.205,2.376,-6.427,1.379


In [15]:
print('Building quarterly farm-anchor stability audit...', flush=True)
anchor_daily_audit = {}
anchor_turbine_quarter_rows = []

for turbine in TRAIN:
    daily = build_anchor_observations(turbine, anchor_stage, labels, theta_star)
    anchor_daily_audit[turbine] = daily
    theta_daily = theta_series[turbine].dropna()
    theta_quarter = theta_daily.groupby(theta_daily.index.to_period('Q')).median()
    work = daily.copy()
    work['quarter_period'] = work.index.to_period('Q')
    for quarter, part in work.groupby('quarter_period'):
        values = part['anchor_deg'].to_numpy(dtype=float)
        weights = part['quality_weight'].to_numpy(dtype=float)
        valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
        if not valid.any():
            continue
        values = values[valid]
        weights = weights[valid]
        base_values = (
            part.loc[valid, 'label_deg'].to_numpy(dtype=float)
            + part.loc[valid, 'state_correction_deg'].to_numpy(dtype=float)
        )
        theta_q = float(theta_quarter.get(quarter, np.nan))
        local_values = base_values + theta_q
        n_eff = float(weights.sum() ** 2 / np.square(weights).sum())
        anchor_turbine_quarter_rows.append({
            'turbine': turbine,
            'farm': FARM_MAP[turbine],
            'quarter': str(quarter),
            'n_days': int(valid.sum()),
            'n_eff_days': n_eff,
            'mean_quality_weight': float(weights.mean()),
            'theta_q_median_deg': theta_q,
            'anchor_fixed_theta_deg': float(np.average(values, weights=weights)),
            'anchor_local_theta_deg': float(np.average(local_values, weights=weights)),
        })
    print(f'  anchor audit: {turbine}', flush=True)

anchor_turbine_quarterly = (
    pd.DataFrame(anchor_turbine_quarter_rows)
    .sort_values(['quarter', 'turbine'])
    .reset_index(drop=True)
)

anchor_quarter_rows = []
for quarter, part in anchor_turbine_quarterly.groupby('quarter'):
    fixed = part['anchor_fixed_theta_deg'].to_numpy(dtype=float)
    local = part['anchor_local_theta_deg'].to_numpy(dtype=float)
    c_fixed = float(np.mean(fixed))
    c_local = float(np.mean(local))
    anchor_quarter_rows.append({
        'quarter': quarter,
        'n_turbines': int(len(part)),
        'C_fixed_theta_deg': c_fixed,
        'C_local_theta_deg': c_local,
        'C_fixed_theta_deviation_deg': c_fixed - float(DEFAULT_ANCHOR_FIT.global_center),
        'turbine_spread_fixed_deg': float(np.max(fixed) - np.min(fixed)),
    })

anchor_quarterly = pd.DataFrame(anchor_quarter_rows).sort_values('quarter').reset_index(drop=True)
b0_quarterly = anchor_turbine_quarterly.merge(
    anchor_quarterly[['quarter', 'C_fixed_theta_deg', 'C_local_theta_deg']],
    on='quarter',
    how='left',
)
b0_quarterly['B0_fixed_theta_deg'] = b0_quarterly['C_fixed_theta_deg'] - b0_quarterly['theta_q_median_deg']
b0_quarterly['B0_local_theta_deg'] = b0_quarterly['C_local_theta_deg'] - b0_quarterly['theta_q_median_deg']

c_fixed_series = anchor_quarterly.set_index('quarter')['C_fixed_theta_deg'].dropna()
c_local_series = anchor_quarterly.set_index('quarter')['C_local_theta_deg'].dropna()
q_x = np.arange(len(c_fixed_series), dtype=float)
c_fixed_slope = float(np.polyfit(q_x, c_fixed_series.to_numpy(dtype=float), 1)[0] * 4.0) if len(c_fixed_series) >= 2 else np.nan
c_local_slope = float(np.polyfit(q_x, c_local_series.to_numpy(dtype=float), 1)[0] * 4.0) if len(c_local_series) >= 2 else np.nan
anchor_stability_summary = pd.DataFrame([{
    'full_fit_C_deg': float(DEFAULT_ANCHOR_FIT.global_center),
    'n_quarters': int(len(anchor_quarterly)),
    'fixed_theta_quarter_mean_deg': float(c_fixed_series.mean()),
    'fixed_theta_quarter_std_deg': float(c_fixed_series.std(ddof=0)),
    'fixed_theta_quarter_range_deg': float(c_fixed_series.max() - c_fixed_series.min()),
    'fixed_theta_max_abs_deviation_deg': float(np.max(np.abs(c_fixed_series - DEFAULT_ANCHOR_FIT.global_center))),
    'fixed_theta_drift_deg_per_year': c_fixed_slope,
    'local_theta_quarter_std_deg': float(c_local_series.std(ddof=0)),
    'local_theta_quarter_range_deg': float(c_local_series.max() - c_local_series.min()),
    'local_theta_drift_deg_per_year': c_local_slope,
    'min_turbines_per_quarter': int(anchor_quarterly['n_turbines'].min()),
}])

display(anchor_stability_summary.round(3))
display(anchor_quarterly.round(3))
display(anchor_turbine_quarterly.round(3))
display(b0_quarterly.round(3))
print('Audit complete: candidate and export variables were not modified.', flush=True)

Building quarterly farm-anchor stability audit...
  anchor audit: PPP_WTG12
  anchor audit: PPP_WTG13
  anchor audit: PPP_WTG14


,full_fit_C_deg,n_quarters,fixed_theta_quarter_mean_deg,fixed_theta_quarter_std_deg,fixed_theta_quarter_range_deg,fixed_theta_max_abs_deviation_deg,fixed_theta_drift_deg_per_year,local_theta_quarter_std_deg,local_theta_quarter_range_deg,local_theta_drift_deg_per_year,min_turbines_per_quarter
0,-6.156,8,-6.073,0.238,0.645,0.492,-0.336,0.556,1.771,0.266,3


,quarter,n_turbines,C_fixed_theta_deg,C_local_theta_deg,C_fixed_theta_deviation_deg,turbine_spread_fixed_deg
0,2023Q1,3,-5.779,-7.395,0.377,1.351
1,2023Q2,3,-5.663,-5.977,0.492,1.149
2,2023Q3,3,-5.884,-5.930,0.271,0.648
3,2023Q4,3,-6.308,-6.646,-0.153,0.461
4,2024Q1,3,-6.237,-5.624,-0.082,0.117
5,2024Q2,3,-6.237,-6.828,-0.082,0.117
6,2024Q3,3,-6.237,-6.570,-0.082,0.117
7,2024Q4,3,-6.237,-5.936,-0.082,0.117


,turbine,farm,quarter,n_days,n_eff_days,mean_quality_weight,theta_q_median_deg,anchor_fixed_theta_deg,anchor_local_theta_deg
0,PPP_WTG12,PPP,2023Q1,34,23.120,0.369,-4.381,-5.043,-5.981
1,PPP_WTG13,PPP,2023Q1,88,64.187,0.413,-2.523,-6.395,-9.269
2,PPP_WTG14,PPP,2023Q1,86,58.008,0.319,-5.391,-5.899,-6.936
3,PPP_WTG12,PPP,2023Q2,34,21.648,0.349,-3.416,-5.066,-5.039
4,PPP_WTG13,PPP,2023Q2,91,67.113,0.450,-3.429,-6.216,-9.996
5,PPP_WTG14,PPP,2023Q2,85,56.852,0.393,-1.542,-5.708,-2.895
6,PPP_WTG12,PPP,2023Q3,71,61.459,0.671,-2.527,-6.053,-5.136
7,PPP_WTG13,PPP,2023Q3,90,76.022,0.576,0.362,-5.476,-5.465
8,PPP_WTG14,PPP,2023Q3,79,67.454,0.603,-5.420,-6.124,-7.190
9,PPP_WTG12,PPP,2023Q4,67,59.912,0.754,-4.412,-6.107,-7.075


,turbine,farm,quarter,n_days,n_eff_days,mean_quality_weight,theta_q_median_deg,anchor_fixed_theta_deg,anchor_local_theta_deg,C_fixed_theta_deg,C_local_theta_deg,B0_fixed_theta_deg,B0_local_theta_deg
0,PPP_WTG12,PPP,2023Q1,34,23.120,0.369,-4.381,-5.043,-5.981,-5.779,-7.395,-1.398,-3.014
1,PPP_WTG13,PPP,2023Q1,88,64.187,0.413,-2.523,-6.395,-9.269,-5.779,-7.395,-3.256,-4.873
2,PPP_WTG14,PPP,2023Q1,86,58.008,0.319,-5.391,-5.899,-6.936,-5.779,-7.395,-0.388,-2.004
3,PPP_WTG12,PPP,2023Q2,34,21.648,0.349,-3.416,-5.066,-5.039,-5.663,-5.977,-2.247,-2.560
4,PPP_WTG13,PPP,2023Q2,91,67.113,0.450,-3.429,-6.216,-9.996,-5.663,-5.977,-2.235,-2.548
5,PPP_WTG14,PPP,2023Q2,85,56.852,0.393,-1.542,-5.708,-2.895,-5.663,-5.977,-4.122,-4.435
6,PPP_WTG12,PPP,2023Q3,71,61.459,0.671,-2.527,-6.053,-5.136,-5.884,-5.930,-3.357,-3.404
7,PPP_WTG13,PPP,2023Q3,90,76.022,0.576,0.362,-5.476,-5.465,-5.884,-5.930,-6.247,-6.293
8,PPP_WTG14,PPP,2023Q3,79,67.454,0.603,-5.420,-6.124,-7.190,-5.884,-5.930,-0.464,-0.510
9,PPP_WTG12,PPP,2023Q4,67,59.912,0.754,-4.412,-6.107,-7.075,-6.308,-6.646,-1.896,-2.234


Audit complete: candidate and export variables were not modified.


## 12. Theta-star mode and shrinkage sensitivity

The rolling estimator sometimes switches between discrete theta modes. This section tests whether a dominant mode is more useful than the current all-window median, and whether an unstable turbine should be pulled toward a robust same-farm reference.

Only the absolute theta reference changes here. The corrected-mean farm anchor, frozen state bundles, soft quality weights, partial amplitude, and beta=1 remain fixed.

In [16]:
THETA_MODE_BAND_DEG = 1.0
print(f'Building theta variants with mode band {THETA_MODE_BAND_DEG:.1f} deg...', flush=True)
THETA_VARIANTS, theta_variant_diagnostics = stabilize_theta_star_map(
    theta_series,
    theta_star,
    apply_ids=TRAIN + TARGETS,
    reference_ids=TRAIN,
    farm_map=FARM_MAP,
    mode_band_deg=THETA_MODE_BAND_DEG,
)
display(theta_variant_diagnostics.round(3))
print('Variants: global_median=current model; main_mode=direct mode; mode_shrink=conservative reference shrinkage.', flush=True)

Building theta variants with mode band 1.0 deg...


,turbine,farm,theta_global_median_deg,theta_main_mode_deg,theta_reference_deg,theta_mode_fraction,theta_mode_mad_deg,mode_confidence,mode_weight,theta_mode_shrink_deg
0,PPP_WTG12,PPP,-3.444,-3.436,-3.444,0.530,0.922,0.600,0.480,-3.440
1,PPP_WTG13,PPP,0.352,0.479,-3.444,0.541,0.224,0.636,0.509,-1.448
2,PPP_WTG14,PPP,-4.354,-3.468,-3.444,0.373,0.915,0.077,0.062,-3.445
3,PPP_WTG17,PPP,-2.483,-2.497,-3.444,0.449,0.071,0.331,0.264,-3.193
4,SSS_WTG06,SSS,-1.513,1.377,-3.444,0.293,0.884,0.000,0.000,-3.444


Variants: global_median=current model; main_mode=direct mode; mode_shrink=conservative reference shrinkage.


In [17]:
print('Running theta-only strict turbine LOTO...', flush=True)
theta_loto, theta_loto_diagnostics = run_theta_anchor_loto(
    TRAIN,
    anchor_stage,
    labels,
    theta_series,
    theta_star,
    farm_map=FARM_MAP,
    shrink_kappa=ANCHOR_SHRINK_KAPPA,
    beta=1.0,
    mode_band_deg=THETA_MODE_BAND_DEG,
)
theta_loto_macro = theta_loto.groupby('theta_variant')[['mae', 'rmse', 'constant_mae', 'constant_rmse']].mean()
display(theta_loto.round(3))
display(theta_loto_macro.round(3))

base = theta_loto_macro.loc['global_median']
for variant in ['main_mode', 'mode_shrink']:
    mae_change = 1.0 - theta_loto_macro.loc[variant, 'mae'] / base['mae']
    rmse_change = 1.0 - theta_loto_macro.loc[variant, 'rmse'] / base['rmse']
    print(f'{variant}: MAE change={mae_change:+.1%}; RMSE change={rmse_change:+.1%}', flush=True)
display(theta_loto_diagnostics.round(3))

Running theta-only strict turbine LOTO...


,theta_variant,holdout,farm,C,B0,mae,rmse,constant_mae,constant_rmse,states
0,global_median,PPP_WTG12,PPP,-6.187,-2.744,0.357,0.604,0.357,0.604,1
1,main_mode,PPP_WTG12,PPP,-5.681,-2.244,0.528,0.543,0.528,0.543,1
2,mode_shrink,PPP_WTG12,PPP,-5.602,-2.911,0.500,0.707,0.500,0.707,1
3,global_median,PPP_WTG13,PPP,-6.115,-6.467,0.427,0.663,1.043,1.367,3
4,main_mode,PPP_WTG13,PPP,-5.669,-6.148,0.653,0.800,0.944,1.439,3
5,mode_shrink,PPP_WTG13,PPP,-5.991,-4.320,2.309,2.392,2.334,2.675,3
6,global_median,PPP_WTG14,PPP,-6.164,-1.809,0.175,0.262,0.175,0.262,1
7,main_mode,PPP_WTG14,PPP,-6.096,-2.628,0.866,0.903,0.866,0.903,1
8,mode_shrink,PPP_WTG14,PPP,-6.103,-4.437,2.675,2.688,2.675,2.688,1


,mae,rmse,constant_mae,constant_rmse
theta_variant,,,,
global_median,0.320,0.510,0.525,0.744
main_mode,0.682,0.749,0.779,0.962
mode_shrink,1.828,1.929,1.836,2.023


main_mode: MAE change=-113.3%; RMSE change=-47.0%
mode_shrink: MAE change=-471.6%; RMSE change=-278.5%


,turbine,farm,theta_global_median_deg,theta_main_mode_deg,theta_reference_deg,theta_mode_fraction,theta_mode_mad_deg,mode_confidence,mode_weight,theta_mode_shrink_deg,holdout,fit_reference
0,PPP_WTG12,PPP,-3.444,-3.436,-2.001,0.530,0.922,0.600,0.480,-2.690,PPP_WTG12,"PPP_WTG13,PPP_WTG14"
1,PPP_WTG13,PPP,0.352,0.479,-2.001,0.541,0.224,0.636,0.509,-0.739,PPP_WTG12,"PPP_WTG13,PPP_WTG14"
2,PPP_WTG14,PPP,-4.354,-3.468,-2.001,0.373,0.915,0.077,0.062,-2.092,PPP_WTG12,"PPP_WTG13,PPP_WTG14"
3,PPP_WTG12,PPP,-3.444,-3.436,-3.899,0.530,0.922,0.600,0.480,-3.677,PPP_WTG13,"PPP_WTG12,PPP_WTG14"
4,PPP_WTG13,PPP,0.352,0.479,-3.899,0.541,0.224,0.636,0.509,-1.671,PPP_WTG13,"PPP_WTG12,PPP_WTG14"
5,PPP_WTG14,PPP,-4.354,-3.468,-3.899,0.373,0.915,0.077,0.062,-3.872,PPP_WTG13,"PPP_WTG12,PPP_WTG14"
6,PPP_WTG12,PPP,-3.444,-3.436,-1.546,0.530,0.922,0.600,0.480,-2.454,PPP_WTG14,"PPP_WTG12,PPP_WTG13"
7,PPP_WTG13,PPP,0.352,0.479,-1.546,0.541,0.224,0.636,0.509,-0.515,PPP_WTG14,"PPP_WTG12,PPP_WTG13"
8,PPP_WTG14,PPP,-4.354,-3.468,-1.546,0.373,0.915,0.077,0.062,-1.665,PPP_WTG14,"PPP_WTG12,PPP_WTG13"


In [18]:
theta_deployment_rows = []
for variant, theta_variant in THETA_VARIANTS.items():
    fit = fit_farm_anchor(
        TRAIN,
        anchor_stage,
        labels,
        theta_variant,
        farm_map=FARM_MAP,
        method='mean',
        shrink_kappa=ANCHOR_SHRINK_KAPPA,
    )
    for turbine in TRAIN + TARGETS:
        centre = fit.center_for(turbine, farm_map=FARM_MAP)
        theta_deployment_rows.append({
            'theta_variant': variant,
            'turbine': turbine,
            'farm': FARM_MAP[turbine],
            'C': centre,
            'theta_star': theta_variant[turbine],
            'B0': centre - theta_variant[turbine],
        })
theta_deployment = pd.DataFrame(theta_deployment_rows).sort_values(['theta_variant', 'turbine']).reset_index(drop=True)
display(theta_deployment.round(3))
print('Theta sensitivity complete: current candidate variables were not modified.', flush=True)

,theta_variant,turbine,farm,C,theta_star,B0
0,global_median,PPP_WTG12,PPP,-6.156,-3.444,-2.712
1,global_median,PPP_WTG13,PPP,-6.156,0.352,-6.507
2,global_median,PPP_WTG14,PPP,-6.156,-4.354,-1.801
3,global_median,PPP_WTG17,PPP,-6.156,-2.483,-3.672
4,global_median,SSS_WTG06,SSS,-6.156,-1.513,-4.643
5,main_mode,PPP_WTG12,PPP,-6.069,-3.436,-2.633
6,main_mode,PPP_WTG13,PPP,-6.069,0.479,-6.549
7,main_mode,PPP_WTG14,PPP,-6.069,-3.468,-2.601
8,main_mode,PPP_WTG17,PPP,-6.069,-2.497,-3.573
9,main_mode,SSS_WTG06,SSS,-6.069,1.377,-7.446


Theta sensitivity complete: current candidate variables were not modified.


## 13. Block-bootstrap uncertainty for theta_star

This is a Monte Carlo uncertainty audit, not a new deployment model. Whole quarters are resampled so the temporal dependence inside the rolling theta estimates is preserved. The bootstrap compares the ordinary mean, median, 10% trimmed mean, and equal-quarter mean.

The final table propagates theta uncertainty through the existing corrected-mean anchor into C and target B0. Labels and relative-state bundles remain fixed.

In [19]:
THETA_BOOTSTRAP_N = 1000
THETA_BOOTSTRAP_SEED = 20260918
theta_bootstrap = {}
theta_bootstrap_rows = []

for i, turbine in enumerate(TRAIN + TARGETS):
    point = theta_location_estimates(theta_series[turbine])
    samples = bootstrap_theta_locations(
        theta_series[turbine],
        n_boot=THETA_BOOTSTRAP_N,
        block_freq='Q',
        trim_fraction=0.10,
        random_state=THETA_BOOTSTRAP_SEED + 1009 * i,
    )
    theta_bootstrap[turbine] = samples
    for method in samples.columns:
        theta_bootstrap_rows.append({
            'turbine': turbine,
            'farm': FARM_MAP[turbine],
            'method': method,
            'point_estimate_deg': point[method],
            'bootstrap_mean_deg': float(samples[method].mean()),
            'bootstrap_std_deg': float(samples[method].std(ddof=1)),
            'p05_deg': float(samples[method].quantile(0.05)),
            'p50_deg': float(samples[method].quantile(0.50)),
            'p95_deg': float(samples[method].quantile(0.95)),
        })

theta_bootstrap_summary = (
    pd.DataFrame(theta_bootstrap_rows)
    .sort_values(['turbine', 'method'])
    .reset_index(drop=True)
)
display(theta_bootstrap_summary.round(3))
print(f'Bootstrap complete: {THETA_BOOTSTRAP_N} quarter-block resamples per turbine.', flush=True)

,turbine,farm,method,point_estimate_deg,bootstrap_mean_deg,bootstrap_std_deg,p05_deg,p50_deg,p95_deg
0,PPP_WTG12,PPP,daily_mean,-3.159,-3.160,0.167,-3.434,-3.158,-2.893
1,PPP_WTG12,PPP,global_median,-3.444,-3.448,0.185,-3.485,-3.447,-3.414
2,PPP_WTG12,PPP,quarter_balanced_mean,-3.188,-3.185,0.176,-3.469,-3.186,-2.905
3,PPP_WTG12,PPP,trimmed_mean,-3.310,-3.318,0.196,-3.646,-3.310,-3.021
4,PPP_WTG13,PPP,daily_mean,-0.469,-0.462,0.434,-1.208,-0.424,0.169
5,PPP_WTG13,PPP,global_median,0.352,0.203,0.444,-0.580,0.357,0.407
6,PPP_WTG13,PPP,quarter_balanced_mean,-0.473,-0.465,0.431,-1.198,-0.427,0.156
7,PPP_WTG13,PPP,trimmed_mean,-0.221,-0.264,0.450,-1.069,-0.197,0.347
8,PPP_WTG14,PPP,daily_mean,-3.667,-3.657,0.462,-4.381,-3.672,-2.880
9,PPP_WTG14,PPP,global_median,-4.354,-4.047,0.608,-5.380,-4.367,-3.418


Bootstrap complete: 1000 quarter-block resamples per turbine.


In [20]:
print('Running theta-location strict turbine LOTO...', flush=True)
theta_location_loto = run_theta_location_loto(
    TRAIN,
    anchor_stage,
    labels,
    theta_series,
    farm_map=FARM_MAP,
    shrink_kappa=ANCHOR_SHRINK_KAPPA,
    beta=1.0,
    trim_fraction=0.10,
)
theta_location_macro = theta_location_loto.groupby('theta_location')[['mae', 'rmse', 'constant_mae', 'constant_rmse']].mean()
display(theta_location_loto.round(3))
display(theta_location_macro.round(3))

base = theta_location_macro.loc['global_median']
for method in ['daily_mean', 'trimmed_mean', 'quarter_balanced_mean']:
    mae_change = 1.0 - theta_location_macro.loc[method, 'mae'] / base['mae']
    rmse_change = 1.0 - theta_location_macro.loc[method, 'rmse'] / base['rmse']
    print(f'{method}: MAE change={mae_change:+.1%}; RMSE change={rmse_change:+.1%}', flush=True)

Running theta-location strict turbine LOTO...


,theta_location,holdout,C,B0,mae,rmse,constant_mae,constant_rmse,states
0,global_median,PPP_WTG12,-6.187,-2.744,0.357,0.604,0.357,0.604,1
1,global_median,PPP_WTG13,-6.115,-6.467,0.427,0.663,1.043,1.367,3
2,global_median,PPP_WTG14,-6.164,-1.809,0.175,0.262,0.175,0.262,1
3,daily_mean,PPP_WTG12,-6.254,-3.095,0.673,0.845,0.673,0.845,1
4,daily_mean,PPP_WTG13,-5.629,-5.161,1.505,1.598,1.555,1.997,3
5,daily_mean,PPP_WTG14,-6.432,-2.765,1.003,1.035,1.003,1.035,1
6,trimmed_mean,PPP_WTG12,-6.245,-2.934,0.521,0.723,0.521,0.723,1
7,trimmed_mean,PPP_WTG13,-5.819,-5.599,1.114,1.210,1.165,1.702,3
8,trimmed_mean,PPP_WTG14,-6.383,-2.487,0.727,0.770,0.727,0.770,1
9,quarter_balanced_mean,PPP_WTG12,-6.264,-3.076,0.655,0.830,0.655,0.830,1


,mae,rmse,constant_mae,constant_rmse
theta_location,,,,
daily_mean,1.060,1.159,1.077,1.292
global_median,0.320,0.510,0.525,0.744
quarter_balanced_mean,1.048,1.148,1.065,1.282
trimmed_mean,0.787,0.901,0.804,1.065


daily_mean: MAE change=-231.5%; RMSE change=-127.5%
trimmed_mean: MAE change=-146.2%; RMSE change=-76.8%
quarter_balanced_mean: MAE change=-227.8%; RMSE change=-125.4%


In [21]:
# Propagate theta-only bootstrap variation through the current corrected-mean C.
base_anchor_means = {}
for turbine in TRAIN:
    daily = build_anchor_observations(turbine, anchor_stage, labels, theta_star)
    values = daily['anchor_deg'].to_numpy(dtype=float) - theta_star[turbine]
    weights = daily['quality_weight'].to_numpy(dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    base_anchor_means[turbine] = float(np.average(values[valid], weights=weights[valid]))

bootstrap_b0_rows = []
location_methods = ['global_median', 'daily_mean', 'trimmed_mean', 'quarter_balanced_mean']
for method in location_methods:
    for b in range(THETA_BOOTSTRAP_N):
        theta_b = {
            turbine: float(theta_bootstrap[turbine].iloc[b][method])
            for turbine in TRAIN + TARGETS
        }
        c_b = corrected_mean_anchor_center(base_anchor_means, theta_b, TRAIN)
        for turbine in TARGETS:
            bootstrap_b0_rows.append({
                'method': method,
                'draw': b,
                'turbine': turbine,
                'C_deg': c_b,
                'theta_deg': theta_b[turbine],
                'B0_deg': c_b - theta_b[turbine],
            })

bootstrap_b0 = pd.DataFrame(bootstrap_b0_rows)
bootstrap_b0_summary = (
    bootstrap_b0.groupby(['method', 'turbine'])[['C_deg', 'theta_deg', 'B0_deg']]
    .agg(['mean', 'std', lambda x: x.quantile(0.05), 'median', lambda x: x.quantile(0.95)])
)
bootstrap_b0_summary.columns = ['_'.join(str(part) for part in column).replace('<lambda_0>', 'p05').replace('<lambda_1>', 'p95') for column in bootstrap_b0_summary.columns]
display(bootstrap_b0_summary.round(3))
print('Bootstrap B0 propagation complete: candidate variables were not modified.', flush=True)

C_deg_mean  C_deg_std  C_deg_p05  \
method                turbine                                       
daily_mean            PPP_WTG17      -5.978      0.212     -6.349   
                      SSS_WTG06      -5.978      0.212     -6.349   
global_median         PPP_WTG17      -6.134      0.183     -6.341   
                      SSS_WTG06      -6.134      0.183     -6.341   
quarter_balanced_mean PPP_WTG17      -5.996      0.212     -6.361   
                      SSS_WTG06      -5.996      0.212     -6.361   
trimmed_mean          PPP_WTG17      -6.099      0.239     -6.514   
                      SSS_WTG06      -6.099      0.239     -6.514   

                                 C_deg_median  C_deg_p95  theta_deg_mean  \
method                turbine                                              
daily_mean            PPP_WTG17        -5.975     -5.655          -2.269   
                      SSS_WTG06        -5.975     -5.655          -1.677   
global_median         PPP_WTG17        -6.158     -5.962          -2.300   
                      SSS_WTG06        -6.158     -5.962          -1.563   
quarter_balanced_mean PPP_WTG17        -5.995     -5.667          -2.259   
                      SSS_WTG06        -5.995     -5.667          -1.679   
trimmed_mean          PPP_WTG17        -6.081     -5.753          -2.235   
                      SSS_WTG06        -6.081     -5.753          -1.638   

                                 theta_deg_std  theta_deg_p05  \
method                turbine                                   
daily_mean            PPP_WTG17          0.402         -2.939   
                      SSS_WTG06          0.261         -2.146   
global_median         PPP_WTG17          0.399         -2.510   
                      SSS_WTG06          1.107         -3.413   
quarter_balanced_mean PPP_WTG17          0.396         -2.926   
                      SSS_WTG06          0.261         -2.147   
trimmed_mean          PPP_WTG17          0.470         -3.026   
                      SSS_WTG06          0.318         -2.197   

                                 theta_deg_median  theta_deg_p95  B0_deg_mean  \
method                turbine                                                   
daily_mean            PPP_WTG17            -2.260         -1.600       -3.710   
                      SSS_WTG06            -1.669         -1.278       -4.301   
global_median         PPP_WTG17            -2.483         -1.555       -3.833   
                      SSS_WTG06            -1.512          0.390       -4.571   
quarter_balanced_mean PPP_WTG17            -2.256         -1.624       -3.737   
                      SSS_WTG06            -1.670         -1.279       -4.318   
trimmed_mean          PPP_WTG17            -2.224         -1.475       -3.864   
                      SSS_WTG06            -1.621         -1.148       -4.460   

                                 B0_deg_std  B0_deg_p05  B0_deg_median  \
method                turbine                                            
daily_mean            PPP_WTG17       0.450      -4.470         -3.700   
                      SSS_WTG06       0.348      -4.857         -4.314   
global_median         PPP_WTG17       0.441      -4.636         -3.683   
                      SSS_WTG06       1.119      -6.533         -4.551   
quarter_balanced_mean PPP_WTG17       0.445      -4.485         -3.731   
                      SSS_WTG06       0.348      -4.866         -4.327   
trimmed_mean          PPP_WTG17       0.518      -4.703         -3.856   
                      SSS_WTG06       0.410      -5.103         -4.475   

                                 B0_deg_p95  
method                turbine                
daily_mean            PPP_WTG17      -2.987  
                      SSS_WTG06      -3.741  
global_median         PPP_WTG17      -3.478  
                      SSS_WTG06      -2.804  
quarter_balanced_mean PPP_WTG17      -3.023  
                      SSS_WTG06      -3.751  
trimmed_me

Bootstrap B0 propagation complete: candidate variables were not modified.


## 14. Deployment absolute-anchor confidence

The current point prediction remains unchanged. This table only attaches uncertainty to the absolute anchor using the `global_median` bootstrap. The thresholds are diagnostic labels, not model gates.

In [22]:
B0_HIGH_WIDTH_DEG = 1.5
B0_MEDIUM_WIDTH_DEG = 2.5
current_deployment = candidate_deployment.set_index('turbine')
confidence_rows = []

for turbine in TARGETS:
    sample = bootstrap_b0[
        (bootstrap_b0['method'] == 'global_median')
        & (bootstrap_b0['turbine'] == turbine)
    ]
    b0_values = sample['B0_deg'].to_numpy(dtype=float)
    theta_values = sample['theta_deg'].to_numpy(dtype=float)
    c_values = sample['C_deg'].to_numpy(dtype=float)
    b0_p05, b0_p50, b0_p95 = np.quantile(b0_values, [0.05, 0.50, 0.95])
    theta_p05, theta_p95 = np.quantile(theta_values, [0.05, 0.95])
    c_p05, c_p95 = np.quantile(c_values, [0.05, 0.95])
    width = float(b0_p95 - b0_p05)
    if width <= B0_HIGH_WIDTH_DEG:
        flag = 'high'
    elif width <= B0_MEDIUM_WIDTH_DEG:
        flag = 'medium'
    else:
        flag = 'low'
    source = 'theta_star' if (theta_p95 - theta_p05) > (c_p95 - c_p05) else 'C'
    confidence_rows.append({
        'turbine': turbine,
        'current_theta_star_deg': float(theta_star[turbine]),
        'current_C_deg': float(current_deployment.loc[turbine, 'C']),
        'current_B0_deg': float(current_deployment.loc[turbine, 'B0']),
        'theta_p05_deg': float(theta_p05),
        'theta_p95_deg': float(theta_p95),
        'C_p05_deg': float(c_p05),
        'C_p95_deg': float(c_p95),
        'B0_p05_deg': float(b0_p05),
        'B0_median_deg': float(b0_p50),
        'B0_p95_deg': float(b0_p95),
        'B0_interval_width_deg': width,
        'dominant_uncertainty_source': source,
        'absolute_anchor_confidence': flag,
    })

deployment_uncertainty_summary = pd.DataFrame(confidence_rows).set_index('turbine')
display(deployment_uncertainty_summary.round(3))
print('Confidence flags are diagnostic only; candidate predictions were not modified.', flush=True)

,current_theta_star_deg,current_C_deg,current_B0_deg,theta_p05_deg,theta_p95_deg,C_p05_deg,C_p95_deg,B0_p05_deg,B0_median_deg,B0_p95_deg,B0_interval_width_deg,dominant_uncertainty_source,absolute_anchor_confidence
turbine,,,,,,,,,,,,,
PPP_WTG17,-2.483,-6.156,-3.672,-2.510,-1.555,-6.341,-5.962,-4.636,-3.683,-3.478,1.158,theta_star,high
SSS_WTG06,-1.513,-6.156,-4.643,-3.413,0.390,-6.341,-5.962,-6.533,-4.551,-2.804,3.729,theta_star,low


Confidence flags are diagnostic only; candidate predictions were not modified.


## 15. Interpretation and limitations

The current model is intentionally conservative.

**What the validation supports**

- The long-term B0 prior is already strong for turbines whose yaw level is effectively static.
- Persistent relative-heading states add value when there is real temporal structure, as seen on WTG13.
- Sensor/reference events must be excluded across all detector sources; otherwise a state detector can turn an encoder event into a false yaw correction.
- The available labels do not support learning a complex amplitude mapping, so \($\beta=1$\) remains the preferred physical prior when transition evidence is sparse.

**What is not established**

- Good LOTO performance on three PPP turbines does not prove the magnitude of unlabeled PPP17/SSS06 state corrections.
- SSS06 is a cross-site transfer and remains the highest domain-shift risk.
- The model should therefore be treated as a constant-prior model with sparse corrections, not as a general daily yaw estimator.

## 16. Validation results visualization

In [23]:
# ============================================================
# Blind validation / final-target state summaries
# ============================================================

def prediction_state_summary(turbine, prediction_map):
    pred = prediction_map[turbine].copy()

    pred["date"] = pd.to_datetime(pred.index)
    pred["cluster"] = pred["cluster"].astype(int)

    summary = (
        pred.groupby("cluster")
        .agg(
            start=("date", "min"),
            end=("date", "max"),
            days=("date", "size"),
            yaw_prediction=("prediction", "median"),
            constant_prior=("constant_prior", "median"),
            relative_correction=(
                "relative_prior_correction",
                "median",
            ),
        )
        .reset_index()
    )

    return summary


print("PPP_WTG17 — blind validation prediction")
display(
    prediction_state_summary(
        "PPP_WTG17",
        candidate_predictions,
    ).round(3)
)

print("SSS_WTG06 — blind final-target prediction")
display(
    prediction_state_summary(
        "SSS_WTG06",
        candidate_predictions,
    ).round(3)
)

PPP_WTG17 — blind validation prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_28052\2123560863.py:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-06-24,175,-6.822,-3.672,3.149
1,1,2023-06-25,2023-10-14,112,-7.912,-3.672,4.239
2,2,2023-10-15,2024-01-06,84,-8.015,-3.672,4.343
3,3,2024-01-07,2024-12-31,360,0.191,-3.672,-3.863


SSS_WTG06 — blind final-target prediction


C:\Users\HJL\AppData\Local\Temp\ipykernel_28052\2123560863.py:43: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(3)


,cluster,start,end,days,yaw_prediction,constant_prior,relative_correction
0,0,2023-01-01,2023-05-27,147,-4.004,-4.643,-0.639
1,1,2023-05-28,2023-09-23,119,-10.074,-4.643,5.431
2,2,2023-09-24,2024-10-12,385,-3.489,-4.643,-1.153
3,3,2024-10-13,2024-12-31,80,-3.288,-4.643,-1.355


In [24]:
validation_preview = submission_predictions.copy()

display(validation_preview.head())
display(validation_preview.tail())

assert validation_preview["turbine_id"].eq(EXPORT_TARGET).all()
assert len(validation_preview) == 731
assert validation_preview["date"].nunique() == 731
assert validation_preview["yaw_misalignment_deg"].notna().all()
assert np.isfinite(validation_preview["yaw_misalignment_deg"]).all()

,turbine_id,date,yaw_misalignment_deg,cluster
0,PPP_WTG17,2023-01-01,-6.821635,0
1,PPP_WTG17,2023-01-02,-6.821635,0
2,PPP_WTG17,2023-01-03,-6.821635,0
3,PPP_WTG17,2023-01-04,-6.821635,0
4,PPP_WTG17,2023-01-05,-6.821635,0


,turbine_id,date,yaw_misalignment_deg,cluster
726,PPP_WTG17,2024-12-27,0.19092,3
727,PPP_WTG17,2024-12-28,0.19092,3
728,PPP_WTG17,2024-12-29,0.19092,3
729,PPP_WTG17,2024-12-30,0.19092,3
730,PPP_WTG17,2024-12-31,0.19092,3


In [25]:
MODEL_CONFIG = globals().get("MODEL", globals().get("model_config"))

if MODEL_CONFIG is None:
    raise RuntimeError("MODEL / model_config not found")

%store bundles
%store MODEL_CONFIG

print("Stored PARS FarmAnchor state")

Stored 'bundles' (dict)
Stored 'MODEL_CONFIG' (ModelConfig)
Stored PARS FarmAnchor state


In [26]:
%store bundles
%store MODEL_CONFIG
%store labels
%store theta_star

print("Stored PARS FarmAnchor state")

if "bundles" not in globals():
    raise RuntimeError("bundles not restored")

if "MODEL_CONFIG" not in globals():
    raise RuntimeError("MODEL_CONFIG not restored")

print("Repo:", ROOT)
print("Loaded PARS FarmAnchor bundles:", list(bundles))
print("MODEL_CONFIG:", MODEL_CONFIG)

Stored 'bundles' (dict)
Stored 'MODEL_CONFIG' (ModelConfig)
Stored 'labels' (dict)
Stored 'theta_star' (dict)
Stored PARS FarmAnchor state
Repo: E:\EnergyHacks\github_release
Loaded PARS FarmAnchor bundles: ['PPP_WTG12', 'PPP_WTG13', 'PPP_WTG14', 'PPP_WTG17', 'SSS_WTG06']
MODEL_CONFIG: ModelConfig(beta_prior=1.0, beta_ridge=12.0, beta_min=0.8, beta_max=1.2, min_beta_changes=3, beta_window_days=21, min_event_confidence=0.25, max_event_shrinkage=0.9, site_common_mode_window_days=7, site_common_mode_min_turbines=3)
